# VAE

In [1]:
# Import required packages
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
from hyperparams import dataset, batch_size, device

## Hyperparams

In [2]:
latent_dim = 256 if dataset.startswith("cifar") else 1024
lr = 5e-4
num_epochs = 10
kl_weight = 4 if dataset.startswith("cifar") else 8
save_figures = True

In [3]:
print(dataset, latent_dim, kl_weight)

cifar10 256 4


## VAE Structure

In [4]:
class ConvVAE(nn.Module):
    def __init__(self, img_size=32, latent_dim=256):
        super().__init__()
        self.img_size = img_size
        self.latent_dim = latent_dim

        # ---------- Encoder ----------
        self.enc = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),  # -> img/2
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),# -> img/4
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),# -> img/8
            nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1),# -> img/16
            nn.ReLU()
        )

        feat_dim = (img_size // 16)
        feat_dim = feat_dim * feat_dim * 512

        self.mu = nn.Linear(feat_dim, latent_dim)
        self.logvar = nn.Linear(feat_dim, latent_dim)

        # ---------- Decoder ----------
        self.fc_dec = nn.Linear(latent_dim, feat_dim)

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.view(h.size(0), -1)
        mu = self.mu(h)
        logvar = self.logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(h.size(0), 512, self.img_size//16, self.img_size//16)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_rec = self.decode(z)
        return x_rec, mu, logvar, z

In [5]:
def reconstruction_loss(x, x_hat):
    return F.mse_loss(x_hat, x, reduction="mean")

def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    recon_loss = reconstruction_loss(x, x_hat)
    kld = kl_divergence(mu, logvar) 

    total_loss = recon_loss + beta * kld
    return total_loss, recon_loss, kld    

### Define optimizer and start training.

In [ ]:
model = ConvVAE(img_size=32, latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loader, test_loader = get_data_loaders(dataset=dataset)
class_names = train_loader.dataset.classes
labels = [f"{i}_{name}" for i, name in enumerate(class_names)]
print(labels)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_recon = 0
    total_kld = 0

    for x, y in tqdm(train_loader, leave=False):
        x = x.to(device)

        x_hat, mu, logvar, z = model(x)

        loss, recon, kld = vae_loss(x, x_hat, mu, logvar, beta=kl_weight)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kld += kld.item()

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"Loss={total_loss/len(train_loader):.4f} "
          f"Recon={total_recon/len(train_loader):.4f} "
          f"KLD={total_kld/len(train_loader):.4f}")


100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s] 


['0_airplane', '1_automobile', '2_bird', '3_cat', '4_deer', '5_dog', '6_frog', '7_horse', '8_ship', '9_truck']


Epoch 1/10 Loss=0.7930 Recon=0.7612 KLD=0.0079


Epoch 2/10 Loss=0.7632 Recon=0.7198 KLD=0.0108


Epoch 3/10 Loss=0.7598 Recon=0.7144 KLD=0.0114


...

Epoch 6/10 Loss=0.7322 Recon=0.6862 KLD=0.0115


Epoch 7/10 Loss=0.7249 Recon=0.6788 KLD=0.0115


Epoch 8/10 Loss=0.7232 Recon=0.6758 KLD=0.0118


Epoch 9/10 Loss=0.7219 Recon=0.6742 KLD=0.0119


Epoch 10/10 Loss=0.7213 Recon=0.6730 KLD=0.0121


In [7]:
import random
import torch

def encode_images(x):
    model.eval()
    with torch.no_grad():
        mu, _ = model.encode(x.to(device))
    return mu 

def latent_distance(x1, x2):
    return torch.norm(x1 - x2, dim=-1)


In [ ]:
import torch
from collections import defaultdict

# ------------------------
# assumes:
# vae = ConvVAE(...)
# loader = CIFAR100 dataloader
# device = "cuda" or "cpu"
# ------------------------

num_classes = 10
vae = model
vae = vae.to(device)
vae.eval()


# --------------------------------------------------
# 1) Collect μ (latent mean) grouped by class label
# --------------------------------------------------
def collect_latents_by_class(vae, loader, device):
    latents_by_class = defaultdict(list)

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            # forward pass encoder only
            mu, logvar = vae.encode(x)     # shapes: [B, latent_dim]

            mu = mu.detach().cpu()

            for zi, yi in zip(mu, y):
                latents_by_class[int(yi.item())].append(zi)

    # convert lists to tensors
    for k in latents_by_class:
        latents_by_class[k] = torch.stack(latents_by_class[k])  # [Nk, d]

    return latents_by_class


# --------------------------------------------------
# 2) Centroid–centroid distance matrix
# --------------------------------------------------
def centroid_distance_matrix(latents_by_class, latent_distance):
    C = len(latents_by_class)

    # compute centroids
    centroids = []
    for c in range(C):
        centroids.append(latents_by_class[c].mean(dim=0))
    centroids = torch.stack(centroids)  # [C, d]

    # fill matrix
    mat = torch.zeros((C, C))
    for i in range(C):
        for j in range(C):
            mat[i, j] = latent_distance(centroids[i], centroids[j])

    return mat


# --------------------------------------------------
# 3) True average pairwise class–class distance matrix
# --------------------------------------------------
def avg_l2_between_sets(A, B):
    # A: [N, d], B: [M, d]

    AA = (A * A).sum(dim=1).unsqueeze(1)   # [N,1]
    BB = (B * B).sum(dim=1).unsqueeze(0)   # [1,M]

    AB = A @ B.T                           # [N,M]

    dist_sq = AA + BB - 2 * AB             # [N,M]

    dist = torch.sqrt(torch.clamp(dist_sq, min=1e-8))

    return dist.mean()

def pairwise_average_distance_matrix_l2(latents_by_class):
    C = len(latents_by_class)
    mat = torch.zeros((C, C))

    for i in range(C):
        zi = latents_by_class[i]

        for j in range(i, C):
            zj = latents_by_class[j]

            val = avg_l2_between_sets(zi, zj)

            mat[i, j] = val
            mat[j, i] = val

    return mat


def pairwise_average_distance_matrix(latents_by_class, latent_distance):
    C = len(latents_by_class)
    mat = torch.zeros((C, C))

    for i in range(C):
        zi = latents_by_class[i]      # [Ni, d]

        for j in range(i, C):         # notice: j starts at i
            zj = latents_by_class[j]  # [Nj, d]

            zi_exp = zi.unsqueeze(1)  # [Ni, 1, d]
            zj_exp = zj.unsqueeze(0)  # [1, Nj, d]

            dists = latent_distance(zi_exp, zj_exp)

            val = dists.mean()

            mat[i, j] = val
            mat[j, i] = val           # mirror to bottom-left

    return mat


In [9]:
latents_by_class = collect_latents_by_class(vae, train_loader, device)
centroid_matrix = centroid_distance_matrix(latents_by_class, latent_distance)
pairwise_matrix = pairwise_average_distance_matrix_l2(latents_by_class)


0
1
2
3
4
5
6
7
8
9


In [10]:
import pandas as pd

class_names = train_loader.dataset.classes
labels = [f"{i}_{name}" for i, name in enumerate(class_names)]

def matrix_to_df(matrix, labels):
    """
    matrix: torch tensor [100,100]
    labels: list length 100
    """
    if isinstance(matrix, torch.Tensor):
        matrix = matrix.cpu().numpy()

    df = pd.DataFrame(matrix, index=labels, columns=labels)
    return df


centroid_df = matrix_to_df(centroid_matrix, labels)
pairwise_df = matrix_to_df(pairwise_matrix, labels)

output_path = f"{dataset}_latent_distances.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    centroid_df.to_excel(writer, sheet_name="centroid_distance")
    pairwise_df.to_excel(writer, sheet_name="pairwise_distance")

print(f"Saved Excel file to: {output_path}")


Saved Excel file to: cifar10_latent_distances.xlsx
